# ⑤ 평가 — 네 가지로 나눠 채점
담당: 평가 · 코드: `rag/evaluation/evaluation.py`, `rag/evaluation/ir.py`

04의 같은 answers 파일을 채점합니다. Retrieval은 결정적 IR, Generation은 문자열 일치이며 API 호출이 없습니다. 유료 RAGAS 심판은 faithfulness와 factual_correctness 두 개입니다. Langfuse 키가 있으면 원래 trace에 점수를 붙이고, 키 없이도 모든 경로가 실행됩니다.

| 평가 | 지표 | 저장 파일 |
|---|---|---|
| Retrieval | Recall@k · MRR@k · Evidence Coverage · 핵심 사실 문자열 일치 | retrieval.csv |
| Generation | 답변의 핵심 사실 문자열 일치 | generation.csv |
| Grounding | faithfulness | grounding.csv |
| End-to-End | factual_correctness · 시간 · 토큰 · 비용 | end_to_end.csv |


In [ ]:
from rag.config import ARTIFACTS, load_config, read_jsonl
from rag.evaluation.evaluation import (evaluate_end_to_end, evaluate_generation, evaluate_grounding,
                                       evaluate_retrieval, rerun_retrieval)
from rag.evaluation.ir import component_table, coverage_table, exclusion_report
from rag.retrieval.retrieval import HybridIndex
from rag.usage import usage_summary

cfg = load_config()
ANSWERS = ARTIFACTS / "answers.jsonl"


## 심판 설정
초기값은 config.yaml에서 읽습니다. Grounding·End-to-End 셀을 실행하면 OpenAI API 비용이 발생합니다. dev는 설정 조절에 쓰고, test는 설정을 고정한 뒤 최종 한 번만 실행합니다.


In [ ]:
JUDGE_MODEL = cfg["evaluation"]["judge_model"]                     # 예: "gpt-5-mini"
JUDGE_REASONING_EFFORT = cfg["evaluation"]["judge_reasoning_effort"]  # 예: "low"
JUDGE_MAX_TOKENS = cfg["evaluation"]["judge_max_tokens"]             # 출력·추론 토큰 한도
JUDGE_PRICES = cfg["evaluation"]["judge_usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가

cfg["evaluation"].update(judge_model=JUDGE_MODEL, judge_reasoning_effort=JUDGE_REASONING_EFFORT,
                         judge_max_tokens=JUDGE_MAX_TOKENS, judge_usd_per_1m_tokens=JUDGE_PRICES)
cfg["evaluation"]


## Retrieval — API 없음
A. 패킹 전 top-k 청크(k=5,10,20): 문서 Recall@k는 그 청크들이 속한 골든 문서 수 / 전체 골든 문서 수, 청크 Recall@k는 포함된 골든 청크 수 / 전체 골든 청크 수입니다. MRR@k는 처음 골든 문서/청크를 찾은 청크 위치의 역수(범위 안에 없으면 0)입니다. 문서를 먼저 중복 제거하지 않습니다.

문서·청크 각각 fused/dense/BM25 순위와 exact-code 집합 포함률을 한 표로 비교합니다. 기관별 순위는 실제 검색과 같이 번갈아 섞으며, 비활성 가중치의 성분은 후보가 없습니다. exact code는 k 없는 집합 포함률입니다.

B. 실제 LLM 컨텍스트: Evidence Coverage = 보낸 골든 청크 수 / 전체 골든 청크 수, full_evidence_coverage = 모두 보냈으면 1입니다. 단일 근거와 다중 근거(multi_hop·comparison 또는 복수 근거)를 따로 봅니다. MRR은 다중 근거의 완전성 지표가 아닙니다.

각 수준은 모든 근거에 해당 라벨이 있는 질문만 채점합니다. 일부/미라벨 질문의 주제별 수와 제외율을 점수 옆과 retrieval_exclusions.csv에 표시합니다. 제외율 분모는 답할 수 있는 골든 질문입니다. unanswerable과 골든 근거 없는 YAML은 별도 집계 후 IR에서 제외합니다.

key_facts_in_context는 검색 청크 속 핵심 사실의 문자열 일치 비율이며 의미적 정확성 판정이 아닙니다. 각 행의 retrieval_settings에 실제 검색 설정·활성 청크 파일·k·저장 rewrite 재사용 여부가 있습니다. 누락 필드가 있는 옛 answers는 유료 호출 전에 멈춥니다.


In [ ]:
retrieval = evaluate_retrieval(ANSWERS, cfg)
display(component_table(retrieval, "doc"))
display(component_table(retrieval, "chunk"))
display(coverage_table(retrieval))
display(exclusion_report(read_jsonl(ANSWERS)))
display(retrieval)


## Generation — API 없음
key_facts_in_answer는 핵심 사실 중 답에 들어 있는 문자열 일치 비율입니다. 의미적 정확성은 End-to-End 심판으로 봅니다.


In [ ]:
generation = evaluate_generation(ANSWERS, cfg)
generation

## Grounding
- `faithfulness`: 답의 주장 중 검색한 청크로 뒷받침되는 비율 (RAGAS) — 낮으면 환각

In [ ]:
grounding = evaluate_grounding(ANSWERS, cfg)
grounding

## End-to-End
- `factual_correctness`: 답과 정답의 사실 일치 F1 (RAGAS)
- `latency_s`: 질문 하나의 재작성 + 검색 + 생성 시간(초)
- `input_tokens` · `cached_input_tokens` · `output_tokens` · `reasoning_tokens`: 답변 생성 토큰 (캐시 입력은 입력, 추론은 출력에 포함)
- `rewrite_*_tokens`: 질문 재작성 토큰
- `answer_cost_usd` · `rewrite_cost_usd`: 호출 당시 단가로 계산해 04에서 저장한 비용
- `cost_usd`: rewrite + 답변 생성 예상 비용 합계. OCR·평가 비용은 `api_usage.csv`에서 별도로 확인하세요.

In [ ]:
end_to_end = evaluate_end_to_end(ANSWERS, cfg)
end_to_end

## Judge / RAGAS API 사용량
네 평가 함수가 CSV를 자동 저장합니다. 남은 두 심판의 실제 호출·재시도는 grounding/end_to_end별로 api_usage.csv에 기록합니다. 아래는 누적 사용량이며 실험 비용은 실행 전후 행 범위로 구분하세요. 30문항 dev 표본은 04 rewrite+answer와 05 두 심판의 설정 단가 기반 견적을 확인하고 승인 후 실행합니다. RAGAS와 Judge 비용을 중복해서 더하지 않습니다.


In [ ]:
usage_summary("judge")

## dev 검색만 다시 평가하기 — API 호출 없음
아래 셀은 별도 실험입니다. 저장된 rewrite.question으로 검색을 다시 실행하고 새 패킹 전 후보·컨텍스트 ID·설정을 같은 행 형식으로 저장합니다. 원본 답변은 재생성하지 않으므로 이 파일은 Retrieval만 채점하세요. 변경한 컨텍스트에 기존 답을 붙여 심판을 실행하면 해당 설정의 생성 결과가 아닙니다. Langfuse trace_id는 비워 원래 실행 점수를 덮지 않습니다. 원본 answers는 보존합니다. test에는 이 조절 경로를 사용하지 않습니다.


In [ ]:
experiment_cfg = load_config()
experiment_cfg["retrieval"]["dense_weight"] = 0.5
experiment_index = HybridIndex(ARTIFACTS, experiment_cfg)
RERUN = ARTIFACTS / "retrieval_rerun" / "answers.jsonl"
rerun_retrieval(ANSWERS, RERUN, experiment_index)
rerun_scores = evaluate_retrieval(RERUN, experiment_cfg)
display(component_table(rerun_scores, "doc"))
display(component_table(rerun_scores, "chunk"))
display(coverage_table(rerun_scores))
